In [1]:
from Bio import Entrez
import pandas as pd
import numpy as np
import os
import json
import time
import regex as re
import xmltodict, json
# ElementTree
import xml.etree.ElementTree as ET
from dataclasses import dataclass

from utils.utils import article_machine, fetch_pubmed_data_given_ids, Paper


[nltk_data] Downloading package stopwords to /home/chris/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/chris/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/chris/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# E-Fetch runs with a maximum of ten thousand studies, we need to separate our IdList in chunks 
def fetch_pubmed_data_given_ids_in_chunks(id_list, chunk_size=10000, output_path=None):
    papers_list = []
    for i in range(0, len(id_list), chunk_size):
        print(f"Fetching papers {i} to {i+chunk_size} out of {len(id_list)}")
        chunk = id_list[i:i+chunk_size]

        papers = fetch_pubmed_data_given_ids(chunk)

        # store all the papers in a class
        for i, paper in enumerate (papers['PubmedArticle']):

            medline_citation = paper['MedlineCitation']

            article = medline_citation.get('Article', {})
            journal_info = article.get('Journal', {})
            journal_issue = journal_info.get('JournalIssue', {})
            pub_date = journal_issue.get('PubDate', {})

            # Extract the data from the XML
            id = medline_citation.get('PMID', 'Not Available')
            title = article.get('ArticleTitle', 'Not Available')
            abstract = article.get('Abstract', {}).get('AbstractText', 'Not Available')
            author_list = article.get('AuthorList', [])
            if isinstance(author_list, dict):
                authors = author_list.get('Author', [])
            else:
                authors = author_list

            authors = [f"{author.get('LastName', 'Not Available')} {author.get('Initials', 'Not Available')}" 
                    for author in authors if isinstance(author, dict)]

            journal = journal_info.get('Title', 'Not Available')
            publication_year = pub_date.get('Year', 'Not Available')
            publication_month = pub_date.get('Month', 'Not Available')
            publication_day = pub_date.get('Day', 'Not Available')
            doi = article.get('ELocationID', 'Not Available')
            keyword_list = medline_citation.get('KeywordList', [])
            if keyword_list and isinstance(keyword_list[0], dict):
                keywords = keyword_list[0].get('Keyword', ['Not Available'])
            else:
                keywords = ['Not Available']
            language_list = article.get('Language', ['Not Available'])

            # Create a Paper object
            paper_object = Paper(id, title, abstract, authors, journal, publication_year, publication_month, publication_day, doi, keywords, language_list)

            # Append the paper object to the list
            papers_list.append(paper_object)

            if output_path:
                if not os.path.exists(output_path):
                    os.makedirs(output_path, exist_ok=True)

                # check if the file already exists and that the id is writable
                filename = f"{output_path}/{id}.json"
                if not os.path.exists(filename) and id.isdigit():
                    paper_object.save_to_json(filename)

    return papers_list


In [3]:
# List of journals to include
journals_to_include = [
    "ACS Applied Materials & Interfaces",
    "ACS Nano",
    "Advanced Functional Materials",
    "Advanced Materials",
    "Angewandte Chemie",
    "Biology and Medicine",
    "Biomaterials",
    "Cell",
    "Clinical Cancer Research",
    "Frontiers in Nanotechnology",
    "Immunity",
    "International Journal of Nanomedicine",
    "Journal of Controlled Release",
    "Journal of Materials Chemistry B",
    "Matter",
    "Molecular Therapy",
    "Nano Letters",
    "Nano Micro Small",
    "Nano Research",
    "Nanomedicine",
    "Nanomedicine: Nanotechnology",
    "Nanoscale",
    "Nature",
    "Nature Biomedical Engineering",
    "Nature Cancer",
    "Nature Communications",
    "Nature Materials",
    "Nature Medicine",
    "Nature Nanotechnology",
    "NPG Asia Materials",
    "Pharmaceutics",
    "PNAS",
    "Science",
    "Science Advances",
    "Science Translational Medicine",
    "Scientific Reports",
    "Small"
]

# journals as small letters
journals_to_include = [journal.lower() for journal in journals_to_include]

# Inclusion criteria
keywords_inclusion = ["nanoparticles", "nanoparticle", "in vivo", "mouse model", "animal model"]

# Exclusion criteria
keywords_exclusion = ["review"]

In [4]:
# compile the query
base_query = f"({keywords_inclusion[0]} OR {keywords_inclusion[1]}) AND ({keywords_inclusion[2]} OR {keywords_inclusion[3]} OR {keywords_inclusion[4]})"

if False:
    # add the exclusion criteria
    query += f" AND NOT ({keywords_exclusion[0]})"

    # add the journals to include
    query += f" AND ("
    for journal in journals_to_include:
        query += f"source:{journal} OR "
    query = query[:-4]
    query += ")"

if False:
    # add the years to include 2015-2021
    query += f" AND (2015/01/01[Date - Publication]:2021/12/31[Date - Publication])"



In [5]:
base_query

'(nanoparticles OR nanoparticle) AND (in vivo OR mouse model OR animal model)'

In [6]:
# create a list of years from 1995 to 2026
years = list(range(1995, 2026))

In [7]:
years

[1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024,
 2025]

In [8]:
for year in years:
    query = base_query + f" AND ({year}/01/01[Date - Publication]:{year}/12/31[Date - Publication])"
    print(f"Querying for year {year}")
    print(f"Query: {query}")

    # search for the query
    id_list = article_machine(query)

    # print the number of results
    print(f"Number of results: {len(id_list)}")

    # fetch the data
    papers = fetch_pubmed_data_given_ids_in_chunks(id_list, output_path="papers")
    

Querying for year 1995
Query: (nanoparticles OR nanoparticle) AND (in vivo OR mouse model OR animal model) AND (1995/01/01[Date - Publication]:1995/12/31[Date - Publication])
Total matching articles: 19

Fetching 1 to 19 out of 19

Retrieved 19 article IDs
Number of results: 19
Fetching papers 0 to 10000 out of 19
Querying for year 1996
Query: (nanoparticles OR nanoparticle) AND (in vivo OR mouse model OR animal model) AND (1996/01/01[Date - Publication]:1996/12/31[Date - Publication])
Total matching articles: 21

Fetching 1 to 21 out of 21

Retrieved 21 article IDs
Number of results: 21
Fetching papers 0 to 10000 out of 21
Querying for year 1997
Query: (nanoparticles OR nanoparticle) AND (in vivo OR mouse model OR animal model) AND (1997/01/01[Date - Publication]:1997/12/31[Date - Publication])
Total matching articles: 21

Fetching 1 to 21 out of 21

Retrieved 21 article IDs
Number of results: 21
Fetching papers 0 to 10000 out of 21
Querying for year 1998
Query: (nanoparticles OR nano

In [ ]:
len(os.listdir("papers"))

In [ ]:
len(os.listdir(r"/home/chris/Data/Projects/playground/nanotech_embed/embeddings"))